**Student Name:** Yaron Winter

**Assignment Overview**

In this assignment, you will explore two fundamental aspects of modern NLP systems: fine-tuning large language models and understanding the attention mechanism that powers them.

**Learning Goals**

By the end of this assignment, you should be able to:

* Understand when and why to fine-tune language models
* Apply QLoRA for efficient adaptation of large models
* Analyze model outputs and limitations
* Explain and implement the attention mechanism
* Interpret attention patterns and their behavior

**Important Note**

* Do not modify or delete the task structure.
* Complete each task with:
   - Clean, well-organized code
   - Relevant visualizations
   - Clear insights and explanations
* Make sure to answer all required questions.
**Submission requirements:**
* Submit a **fully executed notebook** (all cells must run and outputs should be visible).
* There is no need to attach the training and test datasets as files, but you must present them as DataFrame tables within the notebook.


## Install Required Packages and Dataset

In [2]:
#!pip install -q google-colab
!pip install -q bitsandbytes trl
!pip install -q fastapi uvicorn openai
!pip install -q datasets
!pip install -q openai tqdm
!pip install -U trl
!pip install -U torchao
!pip install -q peft

In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline,
    logging,
)
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model,PeftModel
from trl import SFTTrainer
import torch
import json
from datasets import Dataset, load_dataset
import pandas as pd


import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity(logging.CRITICAL)
print("Done.")

Done.


In [4]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

cpu


## **Part 1 — QLoRA Fine-Tuning for Level-Adaptive Question Answering (75 points)**

In this part, you will fine-tune pretrained language models to answer questions at different explanation levels:

* Child — simple and intuitive explanation
* Student — clear educational explanation with moderate technical detail
* Expert — precise, technical, and domain-specific explanation

You will use a subset of the Databricks Dolly 15K dataset.

In [5]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:15000]")

In [6]:
def is_good_question(text):
    return any(q in text.lower() for q in [
        "why", "how", "what is", "explain"
    ])

filtered = [
    ex for ex in dataset
    if ex["category"] == "open_qa" and is_good_question(ex["instruction"])
]

In [7]:
len(filtered), filtered[0]

(1635,
 {'instruction': 'Why can camels survive for long without water?',
  'context': '',
  'response': 'Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.',
  'category': 'open_qa'})

### **Task 1.1 — Prepare a Custom Dataset (15 points)**

Create your own instruction-tuning dataset based on a subset of databricks-dolly-15k.

For each selected question-answer pair, create versions of the answer adapted to the requested expertise level.

Example format:

    Question: What is gradient descent?
    Expertise level: child
    Answer: Gradient descent is like walking downhill step by step until you reach the lowest point.

You should prepare:

1. Training set for fine-tuning -

   Use the filtered Dolly dataset as your source of questions.

   For each question, create examples in the example format per level.

2. Test set for evaluation -
    
   Create a small test set of 20 question-answer examples.

   The test set must:

  * include examples from all three expertise levels
  * be separate from the training set
  * be manually reviewed by you for quality
  * include answers that are appropriate for the requested expertise level

  Recommendation:

  1. Save the generated dataset as a '.jsonl' file so it can be loaded and reused later.

  2. Decide on the training format according to the model architecture:
    
    - For causal language models, such as `HuggingFaceTB/SmolLM2-360M-Instruct`, use one full text field:

    ```text
    ### Question:
    ...

    ### Expertise level:
    child / student / expert

    ### Answer:
    ...```

    - For seq2seq models, such as google/flan-t5-small, separate the input and target:

    input: Question + expertise level
    target: Answer
   
  3. Before generating the full dataset, test the pipeline on a small batch of examples to verify that:

  * the LLM returns valid JSON,
  * each question receives three expertise-level answers,
  * the saved .jsonl file can be loaded correctly,
  * the format matches the training code.

In [9]:
# Get Nebius API Key
from getpass import getpass
#from google.colab import userdata
import os

def get_api_key(name="NEBIUS_API_KEY"):
    api_key = None
    try:
        api_key = os.environ[name]
        print("API Key is taken from the environmental parameters")
    except:
        try:
            api_key = userdata.get(name)
            print("API Key is taken from the google colab user data")
        except:
            try:
                api_key = getpass("Enter API key: ")
                os.environ[name] = api_key
                print("API Key is taken from getpass")
            except:
                raise Exception("API Key for NEBIUS_API_KEY could not be found")

    assert api_key is not None, "API Key is None"
    return api_key
print("Done.")

Done.


In [10]:
import os
from openai import OpenAI

from tqdm import tqdm

# The generation model.
# I use this proprietary model, as in preliminary
# tests it performed much better than the hugging face
# models.
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

QUESTION = "Question"
ANSWER = "Answer"
EXPERTISE = "Expertise"
INSTRUCTION = "instruction"

GENERATION_SYS_PROMPT = """
You are an agent who needs to generate answers for a given questions, according
to the expertises, which are given along the questions.
There are three level of expertises, and this how you should generate your
answer, for each one of them:

Expertise Level: Child
Answer Style: use a simple and intuitive explanation

Expertise Level: Student
Answer Style: use clear educational explanation with moderate technical detail

Expertise Level: Expert
Answer Style: use precise, technical, and domain-specific explanation

These would be the responses for the next example question,
based on the requested expertise level:

Question Example: Why can camels survive for long without water?

Child Response: Camels use the fat in their humps to keep them filled with energy
                and hydration for long periods of time.

Student Response: Camels can survive long periods without water because they have
                  adaptations that minimize water loss and maximize efficiency.
                  Their kidneys and intestines conserve water by producing highly
                  concentrated urine and dry feces, and they can tolerate large
                  fluctuations in body temperature to reduce sweating.

Expert Response: Camels exhibit a suite of physiological adaptations that enable
                extreme dehydration tolerance, including highly efficient renal
                concentrating ability and reduced evaporative water loss via adaptive
                heterothermy. Their erythrocytes are oval and highly deformable, allowing
                circulation under increased blood viscosity during dehydration. Fat
                stored in the hump can be oxidized to yield metabolic water, contributing marginally to hydration
"""

def build_prompt(question: str, expertise: str) -> str:
        return f"""
        Answer the following question according to the requested expertise level.

        Question: {question}
        Expertise level: {expertise}

        Please limit your response to no more than 4 sentences,
        and retrieve it as a string.
"""


client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=get_api_key()
)

def generate_response(question: str, expertise: str, model_name=MODEL_NAME) -> dict:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": GENERATION_SYS_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": build_prompt(question, expertise)
                    }
                ]
            }
        ]
    )
    return json.loads(response.to_json())

def generate_dataset(questions: list) -> pd.DataFrame:
    ds = {QUESTION: [], EXPERTISE: [], ANSWER: []}
    expertises = ["child", "student", "expert"]

    for question in tqdm(questions):
        for expertise in expertises:
          answer = generate_response(question=question, expertise=expertise)

          ds[QUESTION].append(question)
          ds[EXPERTISE].append(expertise)
          ds[ANSWER].append(answer["choices"][0]["message"]["content"])

    return pd.DataFrame(ds)

print("Done.")

Enter API key:  ········


API Key is taken from getpass
Done.


In [11]:
# Test the generation pipeline.
questions = [x[INSTRUCTION] for  x in filtered[:3]]
print(questions)
df = generate_dataset(questions=questions)
df.head(10)

['Why can camels survive for long without water?', 'What is a polygon?', 'What is a verb?']


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:45<00:00, 15.29s/it]


,Question,Expertise,Answer
0,Why can camels survive for long without water?,child,"""Camels use the fat in their humps to keep the..."
1,Why can camels survive for long without water?,student,Camels can survive long periods without water ...
2,Why can camels survive for long without water?,expert,Camels exhibit a suite of physiological adapta...
3,What is a polygon?,child,"""A polygon is a shape that has lots of sides! ..."
4,What is a polygon?,student,"""A polygon is a two-dimensional shape with at ..."
5,What is a polygon?,expert,"""A polygon is a two-dimensional geometric figu..."
6,What is a verb?,child,"""A verb is a word that tells us what someone o..."
7,What is a verb?,student,"""A verb is a word that expresses action, occur..."
8,What is a verb?,expert,"""Verbally, a verb is a word that expresses act..."


**Conclusions & Observations from the initial tests:**

Overall, the results seem reasonable.
I must admit, though, that it is very hard to distnguish between
student answers to an expert answers.
In fact, it would be difficult for me to explain how such differences would look
like, or to distinguish between them.

In [12]:
# Generate the datasets.
# I start with 250 training questions.
# Notice that each question will be ganswered
# in three styles, so it induces a train set
# of 750 entries.
print(f"Total filtered dataset size: {len(filtered)}")

NUM_TRAIN_QUESTIONS = 250
train_questions = [x[INSTRUCTION] for  x in filtered[:NUM_TRAIN_QUESTIONS]]

# The test questions are seperated from the training questions, of course.
# Generate more test questions than needed, and later I will select manually
# the most appropraite ones.
test_questions = [x[INSTRUCTION] for  x in filtered[NUM_TRAIN_QUESTIONS: NUM_TRAIN_QUESTIONS + 50]]

train_df = generate_dataset(train_questions)
test_df = generate_dataset(test_questions)

train_df.to_json("dataset_train.jsonl", orient='records', lines=True)
test_df.to_json("dataset_test.jsonl", orient='records', lines=True)
print("Done.")

Total filtered dataset size: 1635


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [08:47<00:00, 10.55s/it]

Done.


### **Task 1.2 — Fine-Tune Models with QLoRA (10 points)**

Fine-tune the models using QLoRA, meaning:

1. Load the base model in 4-bit quantization
2. Add LoRA adapters
3. Train only the adapter parameters

You should repeat the fine-tuning process for both models:

1. HuggingFaceTB/SmolLM2-360M-Instruct
2. google/flan-t5-small

Pay attention each model requires the dataset to be formated diffrently.

### **Task 1.3 — Evaluation (20 points)**

Evaluate both fine-tuned models on your 20-example test set.

For each model, compare:

* outputs before fine-tuning
* outputs after fine-tuning
* whether the answer matches the requested expertise level
* whether the answer is clear and relevant
* whether the answer stays faithful to the question

Evaluate using this table:

|Question|Level|Base Output|Fine-tuned Output|Expected Answer|Better after FT?|Notes|
|-------|-------|-------|-------|-------|-------|-------|


### **Task 1.4 — Model Comparison and Discussion (15 points)**

Compare the performance of the two models:

* Which model adapted better to the expertise levels?
* Which model produced clearer answers?
* Which model followed the requested format better?
* Did one model hallucinate more than the other?

Explain any differences you observe.

In your discussion, consider that:

* SmolLM2-360M-Instruct is a decoder-only instruction model
* flan-t5-small is an encoder-decoder instruction model
* Different architectures may behave differently on instruction-following and text generation tasks

### **Task 1.5 - Conceptual Questions (15 points)**

Answer the following questions:

1. What changed after fine-tuning?
   
   Discuss whether the model became better at adapting its answer to the requested expertise level.
2. Why is QLoRA more memory efficient?
   
   Explain the role of 4-bit quantization and LoRA adapters.
3. What happens if you increase the LoRA rank?
   
   Discuss the tradeoff between model capacity, memory usage, and overfitting risk.
4. Why use LoRA / QLoRA instead of prompt engineering?

      Discuss:

      * In what cases prompt engineering is sufficient
      * When fine-tuning becomes necessary
      * What advantages QLoRA provides over prompting
      * What are the trade-offs (cost, flexibility, control)

## **Part 2 — Understanding Attention (25 points)**

Goal:

Build intuition for how attention works.

Task:

You will implement a simple attention mechanism from scratch (PyTorch) and visualize its behavior.

### **Task 2.1 Implement Scaled Dot-Product Attention (5 points)**

Given:

* Query (Q)
* Key (K)
* Value (V)

Compute:

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implements Scaled Dot-Product Attention.

    Args:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_v, d_v)
        mask: (batch, heads, seq_len_q, seq_len_k) or None

    Returns:
        output: (batch, heads, seq_len_q, d_v)
        attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """


    #  Get dimension for scaling
    d_k = Q.size(-1)


    # Compute attention scores

    scores = # TODO: compute dot-product between Q and K^T


    # Scale the scores

    # TODO: divide scores by sqrt(d_k)


    #  Apply mask (if given)

    if mask is not None:
        # TODO: mask out invalid positions (set to -inf)
        pass


    # Softmax to get attention

    attention_weights = # TODO: apply softmax over last dimension


    # Compute weighted sum

    output = #TODO: multiply attention_weights with V

    return output, attention_weights

### **Task 2.2 Visualize Attention (5 points)**


Plot attention weights as a heatmap for the given sentence


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import math



sentence = "the cat sat on the mat"


tokens = # TODO: split sentence into tokens

seq_len = # TODO: compute sequence length



# Create dummy embeddings

d_model = 16


embeddings = # TODO: initialize random embeddings of shape (seq_len, d_model)

# Treat embeddings as Q, K, V
Q = embeddings
K = embeddings
V = embeddings



# Compute Scaled Dot-Product Attention

attention_weights = # TODO: compute attention weights (use function from above)



# Plot attention heatmap

plt.figure(figsize=(8, 6))

# TODO: plot heatmap using seaborn
# - use attention_weights
# - set xticklabels and yticklabels to tokens
# - choose a colormap (e.g., "viridis")

plt.title("Attention Weights Heatmap")

# TODO: label axes
# plt.xlabel(...)
# plt.ylabel(...)

# TODO: rotate ticks if needed

plt.show()



### **Task 2.3 Experiments (15 points)**

**Experiment 1 — Change One Word:**

1. Use a simple sentence, for example:
    the cat sat on the mat
2. Compute and plot the attention weights as a heatmap.
3. Change one word in the sentence, for example:
    the dog sat on the mat
4. Recompute and plot the attention heatmap.
5. Compare the two heatmaps and explain whether the attention pattern changed.

**Experiment 2 — Compare Different Attention Heads:**

Repeat the attention visualization using at least two different attention heads.

For each head, create separate projection matrices:

$$W_Q, W_K, W_V$$

Use them to compute:
$$Q=XW_Q, K=XW_K, V=XW_V$$

Then compute and plot the attention weights for each head.

**Questions**

Answer briefly:

1. Did changing one word affect the attention weights? Why or why not?
2. Do different attention heads focus on different tokens?
3. Why might multi-head attention be useful in Transformer models?